<a href="https://colab.research.google.com/github/Artem1965/DS23_Module3_Assignment1_Supervised_Learning/blob/main/DS23_Module3_Assignment1_Supervised_Starter_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 3 · Assignment 1 — Supervised Learning
### DS23 · Certified Data Scientist with Agentic AI · by Shlomit Levavi

**This is a guided starter notebook, not a solution.**
It gives you the skeleton of a professional supervised-learning workflow.
Your job is to fill in every `# TODO`, and — more importantly — to *interrogate*
your model and answer the guiding questions in `REPORT.md`.

> The model is the easy 30%. The thinking is the graded 70%.

**Workflow (6 stations):**
0. Frame the problem and pick a success metric
1. Load data, split honestly, build a dumb baseline
2. Train at least 3 model families
3. Evaluate once on the locked test set
4. Interrogate the model (errors, importance, stability)
5. Translate to the real world (Model Card)

Pick **one** task in Part 1. Write all code and comments in **English**.


---
## Part 0 · Frame the problem

Before any code, write in `REPORT.md`:
- The business question in one paragraph.
- The exact target you are predicting.
- The **primary metric** you will optimize, and **why it fits the business cost**
  (what does a false positive cost vs a false negative? for forecasting, what does
  an over- vs under-forecast cost?).

Do not skip this. A model optimized for the wrong metric is worse than no model.


---
## Part 1 · Setup, data, honest split, baseline

In [15]:
from google.colab import drive
drive.mount('/content/drive')

# Core imports. Add what you need; keep everything in English.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.metrics import (
    confusion_matrix, classification_report,
    precision_score, recall_score, f1_score, roc_auc_score,
    RocCurveDisplay, PrecisionRecallDisplay,
    mean_absolute_error, mean_squared_error, r2_score,
)

RANDOM_STATE = 42
pd.set_option("display.max_columns", 50)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Choose ONE task

Uncomment exactly one loader below. Each returns the raw material you need;
**feature selection and engineering choices are yours** (keep it light — heavy
feature engineering is Module 5).

| Option | Type | Target | Notes |
|---|---|---|---|
| A | Binary classification | `review_score <= 2` (negative review) | The course thread. Try a **time-based split**. |
| B | Binary classification | customer churn | Fresh dataset. No class code to copy. |
| C | Forecasting (regression) | next-day order volume | Time series **as a supervised problem** (Session 18). |


In [22]:
# Set the path where your Olist CSVs live (the 9-table Kaggle structure from class).
OLIST_DIR =  '/content/drive/My Drive/BAR ILAN/Modules/Module 1_  AI Foundations and Ethics/2. Python and SQL for Data Science/Data/Brazilian E-Commerce Public Dataset by Olist/'



In [23]:

# ---- Option C · Olist daily order volume (forecasting as supervised) ----------
def load_olist_daily_volume(olist_dir=OLIST_DIR):
    """One row per calendar day with the number of orders placed that day.
    Turned into a supervised problem with lag/rolling features (Session 18)."""
    orders = pd.read_csv(olist_dir + "olist_orders_dataset.csv",
                         parse_dates=["order_purchase_timestamp"])
    daily = (orders.set_index("order_purchase_timestamp")
                   .resample("D").size().rename("n_orders").reset_index())
    daily = daily.rename(columns={"order_purchase_timestamp": "date"})
    # Keep the stable, well-populated period. The launch tail (until early Jan 2017)
# and the truncated end (after Aug 2018) are artifacts of incomplete data capture,
# not real demand — including them would distort baseline and training.
    daily = daily[(daily["date"] >= "2017-01-05") &
              (daily["date"] <= "2018-08-22")].reset_index(drop=True)

    # Lag and rolling features — all use shift(1+) so they rely ONLY on the past.
    daily["lag_1"] = daily["n_orders"].shift(1)
    daily["lag_7"] = daily["n_orders"].shift(7)
    daily["rolling_mean_7"] = daily["n_orders"].shift(1).rolling(7).mean()
    daily["rolling_mean_14"] = daily["n_orders"].shift(1).rolling(14).mean()

    # Calendar features.
    daily["day_of_week"] = daily["date"].dt.dayofweek
    daily["month"] = daily["date"].dt.month
    daily["is_weekend"] = (daily["day_of_week"] >= 5).astype(int)

    # Drop rows with NaN created by the lag/rolling features (first 14 rows).
    daily = daily.dropna().reset_index(drop=True)
    return daily

df = load_olist_daily_volume()
print(df.shape)
print(df[["date", "n_orders"]].head())
print(df[["date", "n_orders"]].tail())

(581, 9)
        date  n_orders
0 2017-01-19        29
1 2017-01-20        29
2 2017-01-21        24
3 2017-01-22        31
4 2017-01-23        39
          date  n_orders
576 2018-08-18       198
577 2018-08-19       204
578 2018-08-20       256
579 2018-08-21       243
580 2018-08-22       187


In [ ]:
# Inspect head and tail to find where stable volume begins/ends
print(df[["date", "n_orders"]].head(120).to_string())
print(df[["date", "n_orders"]].tail(30).to_string())

In [ ]:
print(df[df["date"] >= "2018-08-01"][["date", "n_orders"]].to_string())


### Honest split + dumb baseline

This is the station that answers *"is my model worth anything?"*.

- **Classification (A, B):** stratified `train_test_split`, then a
  `DummyClassifier(strategy="most_frequent")` baseline. Record its score.
- **Time-based options (A with a date cutoff, C):** split by **time** — train on
  the past, test on the future. A random split here *leaks the future* (Session 18).
  Baseline for forecasting = a naive model (e.g. "tomorrow = today" or seasonal naive).

Keep the test set **locked** until Part 3.


In [33]:
# TODO: build X, y from df (your feature choice), then split.
# Classification example shape (fill in):
feature_cols = [c for c in df.columns if c not in ("n_orders", "date")]
X = df[ feature_cols ]
y = df["n_orders"]

# TODO: time-based alternative (Option C, or Option A with a cutoff date):
cutoff = df["date"].quantile(0.8)
train = df[df["date"] <= cutoff]
test  = df[df["date"] >  cutoff]

X_train, y_train = train[feature_cols], train["n_orders"]
X_test,  y_test  = test[feature_cols],  test["n_orders"]

print("train:", train["date"].min(), "→", train["date"].max(), "| n =", len(train))
print("test :", test["date"].min(),  "→", test["date"].max(),  "| n =", len(test))

naive_pred  = X_test["lag_1"]   # random walk
snaive_pred = X_test["lag_7"]   # seasonal naive


snaive_mae  = mean_absolute_error(y_test, snaive_pred)
snaive_rmse = np.sqrt(mean_squared_error(y_test, snaive_pred))
naive_mae  = mean_absolute_error(y_test, naive_pred)
naive_rmse = np.sqrt(mean_squared_error(y_test, naive_pred))

baseline_score = naive_rmse   # use YOUR primary metric, not accuracy by default

print("Seasonal-naive baseline — RMSE:", round(snaive_rmse, 2), "| MAE:", round(snaive_mae, 2))
print("Baseline score (chosen = random-walk RMSE):", round(baseline_score, 2))
print("Random-walk  (lag_1) — RMSE:", round(naive_rmse, 2), "| MAE:", round(naive_mae, 2))

train: 2017-01-19 00:00:00 → 2018-04-28 00:00:00 | n = 465
test : 2018-04-29 00:00:00 → 2018-08-22 00:00:00 | n = 116
Seasonal-naive baseline — RMSE: 64.04 | MAE: 49.79
Baseline score (chosen = random-walk RMSE): 49.09
Random-walk  (lag_1) — RMSE: 49.09 | MAE: 37.38


---
## Part 2 · Build at least three model families

Use the algorithms from Sessions 16-18. Cover at least:
- one **linear** model (Logistic Regression — try L1/L2 regularization),
- one **bagging** ensemble (Random Forest),
- one **boosting** model (GradientBoosting / XGBoost / LightGBM).

Select with **cross-validation** (`StratifiedKFold` for classification). Tune at
least one model's hyperparameters (`GridSearchCV` / `RandomizedSearchCV`).

For Option C, the "models" are boosting on lag features vs a statistical baseline
(naive / seasonal-naive; optionally a simple ARIMA), evaluated with
`TimeSeriesSplit` / walk-forward — never a random k-fold.


In [ ]:
# TODO: instantiate and cross-validate your models. Keep results in a dict.
# from sklearn.linear_model import LogisticRegression
# from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
# import xgboost as xgb  # or lightgbm

models = {
    # "logreg": LogisticRegression(max_iter=1000, ...),
    # "rf": RandomForestClassifier(random_state=RANDOM_STATE, ...),
    # "xgb": xgb.XGBClassifier(...),
}

cv_scores = {}
# for name, model in models.items():
#     scores = cross_val_score(model, X_train, y_train,
#                              cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE),
#                              scoring="f1")   # TODO: your metric
#     cv_scores[name] = (scores.mean(), scores.std())
cv_scores


In [ ]:
# TODO: tune at least one model's hyperparameters with cross-validation.
# from sklearn.model_selection import GridSearchCV
# grid = GridSearchCV(estimator=..., param_grid={...}, scoring=..., cv=5)
# grid.fit(X_train, y_train)
# print(grid.best_params_, grid.best_score_)


---
## Part 3 · Honest evaluation on the locked test set

Touch the test set **once**. Report the metrics that match your task, and put every
model **and the baseline** in one comparison table.

- Classification: confusion matrix, precision, recall, F1, ROC-AUC, PR curve.
- Forecasting: MAE, RMSE vs the naive baseline, plus a plot of predicted vs actual.


In [ ]:
# A small helper to keep your comparison honest and readable.
def results_table(rows):
    """rows: list of dicts, e.g. {'model': 'rf', 'f1': 0.61, 'roc_auc': 0.78}.
    Always include the baseline as one of the rows."""
    return pd.DataFrame(rows).set_index("model").sort_values(
        by=[c for c in ["f1", "roc_auc", "rmse"] if c in (rows[0] if rows else {})][:1] or None
    )

# TODO: evaluate each fitted model on (X_test, y_test) and collect rows.
# rows = [
#     {"model": "baseline", "f1": ..., "roc_auc": ...},
#     {"model": "logreg",   "f1": ..., "roc_auc": ...},
#     ...
# ]
# results_table(rows)


In [ ]:
# TODO: confusion matrix + ROC + PR for your BEST model (classification).
# cm = confusion_matrix(y_test, best_pred)
# print(cm)
# RocCurveDisplay.from_estimator(best_model, X_test, y_test); plt.show()
# PrecisionRecallDisplay.from_estimator(best_model, X_test, y_test); plt.show()


---
## Part 4 · Interrogate the model

This is where data scientists are made. Do all four:

1. **Error analysis** — pull the 5 worst mistakes. What do they share? Data issue or hard case?
2. **Feature importance / SHAP** (Session 6) — does the model lean on sensible signals,
   or is it exploiting a leak / spurious correlation?
3. **Stability** — how much does the score vary across CV folds? Would you trust the number?
4. **Bias-variance read** — compare train vs test/CV. Over- or under-fitting?


In [ ]:
# TODO 1 · Error analysis: find the most confident wrong predictions and inspect them.
# proba = best_model.predict_proba(X_test)[:, 1]
# err = X_test.assign(y_true=y_test.values, p=proba)
# err["wrongness"] = (err["p"] - err["y_true"]).abs()
# err.sort_values("wrongness", ascending=False).head(5)


In [ ]:
# TODO 2 · Feature importance or SHAP.
# import shap
# explainer = shap.TreeExplainer(best_model)
# shap_values = explainer.shap_values(X_test)
# shap.summary_plot(shap_values, X_test)


In [ ]:
# TODO 3 · Stability across folds (report mean and std), and
# TODO 4 · train-vs-test gap for your best model. Write the conclusion in REPORT.md.


---
## Part 5 · Model Card (fill this in, then copy to REPORT.md)

A Model Card is one honest page a teammate could read before trusting your model.
Fill every field. "Unknown" with a reason is an acceptable, professional answer.


In [ ]:
MODEL_CARD = """
# Model Card

## 1. Overview
- Task / business question:
- Dataset (which option) and time range:
- Target definition:

## 2. Metric & performance
- Primary metric and WHY (business cost of FP vs FN / over- vs under-forecast):
- Dumb baseline score:
- Best model score (on the locked test set):
- Did it beat the baseline meaningfully? Is it worth deploying?

## 3. What the model relies on
- Top features and whether they make business sense:
- Any feature you suspect is a leak or spurious? How did you check?

## 4. Limitations & failure modes
- The 5 worst errors — what is the pattern?
- Where would this model break?
- Stability across folds (mean +/- std):

## 5. Fairness / ethics
- Could any group be systematically mis-served by this model?

## 6. Real world
- If deployed Monday: what would you monitor? What triggers a retrain?
- With two more weeks / more data, what would you do next?
"""
print(MODEL_CARD)


---
### Submission checklist
- [ ] This notebook runs **top to bottom** with no errors (Kernel → Restart & Run All).
- [ ] All code and comments are in **English**.
- [ ] A dumb baseline is present and compared in one table.
- [ ] The test set was touched exactly once.
- [ ] `REPORT.md` answers all guiding questions and contains the filled Model Card.

Good luck — and stay skeptical of your own results.
